# Validation: N_SAMPLES=40, N_LOCAL=25

This notebook validates LLM responses from the enforce_knowledge experiment with config:
- **N_SAMPLES:** 40 training + 40 prediction examples
- **N_LOCAL:** 25 SHAP + LIME local explanations

## Validation goals
1. **Feature ranking accuracy** - Compare LLM's top-3 features per class against ground truth SHAP rankings
2. **SHAP citation accuracy** - Check if LLM cites SHAP values correctly (Phase 2)
3. **Fabrication detection** - Flag invented percentages, misclassifications, overvaluation of User_Agent
4. **Phase comparison** - Does Phase 2 (with XAI) improve over Phase 1 (raw data)?

In [ ]:
import json
import pandas as pd
from IPython.display import display, Markdown, HTML
import sys
sys.path.append('.')
from validation_helpers import (
    compute_ground_truth,
    extract_all_rankings,
    extract_shap_values,
    score_shap_citations,
    check_user_agent_overvaluation,
    check_fabricated_percentages,
    ranking_accuracy,
    ranking_top3_set_overlap,
    display_response,
    display_ranking_comparison
)

In [ ]:
# Load results
with open('resultados_enforce_knowledge_samples_40_local_25.json', 'r', encoding='utf-8') as f:
    results = json.load(f)

print(f"Loaded results for {len(results)} models")
print(f"Models: {list(results.keys())}")

In [ ]:
# Compute ground truth
gt = compute_ground_truth('../Network_logs.csv')

print("=== Ground Truth SHAP Rankings (Top-3 per class) ===")
for cls_name, ranking in gt['shap_rankings'].items():
    print(f"{cls_name}: {' > '.join(ranking[:3])}")

print(f"\nModel accuracy: {gt['accuracy']:.4f}")
print(f"Features: {gt['feature_names']}")
print(f"Classes: {gt['class_names']}")

### Ground Truth Reference

**Expected top-3 feature importance per class:**
- **BotAttack:** Port > Status > Payload_Size
- **Normal:** Port > Status > Payload_Size  
- **PortScan:** Payload_Size > Status > Port

**Key insight:** User_Agent has near-zero SHAP importance (< 0.006) despite being commonly overvalued by LLMs.

In [ ]:
# Validation results storage
validation_results = {}

for model_name, model_data in results.items():
    print(f"\n{'='*60}")
    print(f"Model: {model_name}")
    print('='*60)
    
    validation_results[model_name] = {}
    
    for phase_key, phase_label in [
        ('phase1_without_xai', 'Phase 1 (No XAI)'),
        ('phase2_with_xai', 'Phase 2 (With XAI)')
    ]:
        if phase_key not in model_data:
            print(f"  {phase_label}: SKIPPED (not present)")
            continue
        
        text = model_data[phase_key]
        print(f"\n  {phase_label}: {len(text)} chars")
        
        # Extract rankings
        predicted_rankings = extract_all_rankings(text, gt['class_names'])
        
        # Ranking accuracy (position match)
        correct, total, per_class = ranking_accuracy(predicted_rankings, gt['shap_rankings'], gt['class_names'])
        ranking_acc = correct / total if total > 0 else 0
        
        # Set overlap (order-independent)
        overlap_correct, overlap_total = ranking_top3_set_overlap(predicted_rankings, gt['shap_rankings'], gt['class_names'])
        set_overlap = overlap_correct / overlap_total if overlap_total > 0 else 0
        
        # User_Agent overvaluation check
        ua_hits = check_user_agent_overvaluation(text)
        
        # Fabricated percentages (Phase 1 only - no SHAP to back them up)
        if phase_key == 'phase1_without_xai':
            fab_pct = check_fabricated_percentages(text)
        else:
            fab_pct = []
        
        # SHAP citation accuracy (Phase 2 only)
        if phase_key == 'phase2_with_xai':
            cited_shap = extract_shap_values(text, gt['class_names'])
            shap_checks = score_shap_citations(cited_shap, gt['shap_ground_truth'], gt['class_names'])
            n_exact = sum(1 for c in shap_checks if c['status'] == 'EXACT')
            n_close = sum(1 for c in shap_checks if c['status'] == 'CLOSE')
            n_off = sum(1 for c in shap_checks if c['status'] == 'OFF')
        else:
            cited_shap = {}
            shap_checks = []
            n_exact = n_close = n_off = 0
        
        validation_results[model_name][phase_key] = {
            'text_length': len(text),
            'predicted_rankings': predicted_rankings,
            'ranking_accuracy': ranking_acc,
            'set_overlap': set_overlap,
            'per_class': per_class,
            'user_agent_overvaluation': ua_hits,
            'fabricated_percentages': fab_pct,
            'shap_citations': cited_shap if phase_key == 'phase2_with_xai' else None,
            'shap_check_counts': {'exact': n_exact, 'close': n_close, 'off': n_off} if phase_key == 'phase2_with_xai' else None,
        }
        
        print(f"    Ranking accuracy (position match): {ranking_acc:.1%} ({correct}/{total})")
        print(f"    Set overlap (order-independent): {set_overlap:.1%}")
        print(f"    User_Agent overvaluation hits: {len(ua_hits)}")
        if phase_key == 'phase1_without_xai':
            print(f"    Fabricated percentages: {len(fab_pct)}")
        else:
            print(f"    SHAP citations: {n_exact} exact, {n_close} close, {n_off} off")

## Summary Table

In [ ]:
# Build summary DataFrame
summary_rows = []

for model_name, phases in validation_results.items():
    for phase_key, metrics in phases.items():
        summary_rows.append({
            'model': model_name,
            'phase': 'Phase 1 (No XAI)' if phase_key == 'phase1_without_xai' else 'Phase 2 (With XAI)',
            'ranking_accuracy': metrics['ranking_accuracy'],
            'set_overlap': metrics['set_overlap'],
            'user_agent_overvaluation': len(metrics['user_agent_overvaluation']),
            'fabricated_percentages': len(metrics['fabricated_percentages']) if phase_key == 'phase1_without_xai' else 0,
            'shap_exact': metrics['shap_check_counts']['exact'] if phase_key == 'phase2_with_xai' else None,
            'shap_close': metrics['shap_check_counts']['close'] if phase_key == 'phase2_with_xai' else None,
            'shap_off': metrics['shap_check_counts']['off'] if phase_key == 'phase2_with_xai' else None,
            'text_length': metrics['text_length'],
        })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

## Ranking Comparison Tables

In [ ]:
# Display ranking comparison for each model/phase
for model_name, phases in validation_results.items():
    print(f"\n{'='*60}")
    print(f"{model_name}")
    print('='*60)
    
    for phase_key, metrics in phases.items():
        phase_label = 'Phase 1 (No XAI)' if phase_key == 'phase1_without_xai' else 'Phase 2 (With XAI)'
        print(f"\n--- {phase_label} ---")
        display_ranking_comparison(
            model_name, phase_label,
            metrics['predicted_rankings'], gt['shap_rankings'], gt['class_names']
        )

## Detailed Inspection

In [ ]:
# Show flagged issues
for model_name, phases in validation_results.items():
    for phase_key, metrics in phases.items():
        phase_label = 'Phase 1' if phase_key == 'phase1_without_xai' else 'Phase 2'
        
        if metrics['user_agent_overvaluation']:
            print(f"\n{model_name} - {phase_label}: User_Agent overvaluation ({len(metrics['user_agent_overvaluation'])} hits)")
            for hit in metrics['user_agent_overvaluation'][:3]:
                print(f"  - {hit}")
        
        if metrics['fabricated_percentages']:
            print(f"\n{model_name} - {phase_label}: Fabricated percentages ({len(metrics['fabricated_percentages'])} hits)")
            for item in metrics['fabricated_percentages'][:3]:
                print(f"  - {item['feature']}: {item['percentage']}%")
        
        if phase_key == 'phase2_with_xai' and metrics['shap_check_counts']['off'] > 0:
            print(f"\n{model_name} - {phase_label}: SHAP citations off ({metrics['shap_check_counts']['off']} values)")

## Full Responses (for manual inspection)

In [ ]:
# Optionally display full responses (commented out by default - very verbose)
# Uncomment to inspect specific model/phase

# model_to_inspect = "qwen3:14b"
# phase_to_inspect = "phase2_with_xai"
# display_response(
#     model_to_inspect, 
#     phase_to_inspect.replace('_without_xai', 'Without XAI').replace('_with_xai', 'With XAI'),
#     results[model_to_inspect][phase_to_inspect]
# )